In [48]:
import numpy as np
import pandas as pd

In [49]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"])

0

In [50]:
dftrain = pd.read_csv('a.csv')
train = dftrain[dftrain['league'] == 'LEC']
train = train[train['position'] == 'team']
train = train.reset_index(drop = True)

C:\Users\rares\AppData\Local\Temp\ipykernel_25652\2377733350.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  dftrain = pd.read_csv('a.csv')


In [51]:
import pandas as pd

splits = [
    ('LEC 2026 Versus - Team Stats - OraclesElixir.csv',      2026, 1, 'Versus'),
    ('LEC 2026 Spring - Team Stats - OraclesElixir (1).csv',  2026, 2, 'Spring'),
    ('LEC 2025 Winter - Team Stats - OraclesElixir.csv',      2025, 1, 'Winter'),
    ('LEC 2025 Spring - Team Stats - OraclesElixir.csv',      2025, 2, 'Spring'),
    ('LEC 2025 Summer - Team Stats - OraclesElixir.csv',      2025, 3, 'Summer'),
    ('LEC 2024 Winter - Team Stats - OraclesElixir.csv',      2024, 1, 'Winter'),
    ('LEC 2024 Spring - Team Stats - OraclesElixir (1).csv',  2024, 2, 'Spring'),
    ('LEC 2024 Summer - Team Stats - OraclesElixir.csv',      2024, 3, 'Summer'),
]

historical_cols = [
    'Team', 'GP', 'W', 'L',
    'KD', 'CKPM', 'GPR',
    'FBN%', 'BN%', 'ELD%',
    'WPM', 'CWPM', 'WCPM',
    'LNE%', 'JNG%',
]

dfs = []
for path, year, split_num, split_name in splits:
    tmp = pd.read_csv(path)
    tmp = tmp[historical_cols].copy()
    tmp['year'] = year
    tmp['split'] = split_num
    tmp['split_name'] = split_name
    tmp['winrate'] = tmp['W'] / tmp['GP']
    dfs.append(tmp)

split_df = pd.concat(dfs, ignore_index=True)
split_df = split_df.rename(columns={'Team': 'teamname'})

split_df = split_df.sort_values(['year', 'split']).reset_index(drop=True)

print(split_df.shape)
print(split_df.head())

(82, 19)
        teamname  GP  W  L    KD  CKPM   GPR FBN%  BN%  ELD%   WPM  CWPM  \
0         Fnatic   9  6  3  1.18  0.94  1.04  44%  60%    0%  3.43  1.06   
1     G2 Esports   9  6  3  1.06  0.77  0.21  33%  50%   NaN  3.96  1.16   
2         GiantX   9  4  5  0.86  0.77 -0.72  44%  45%   50%  3.77  1.24   
3   Karmine Corp   9  2  7  0.74  0.83 -0.80  33%  42%  100%  3.72  1.31   
4  MAD Lions KOI   9  4  5  1.03  0.84  0.26  44%  50%   NaN  3.09  1.12   

   WCPM   LNE%   JNG%  year  split split_name   winrate  
0  1.72  51.4%  54.6%  2024      1     Winter  0.666667  
1  1.63  51.1%  51.4%  2024      1     Winter  0.666667  
2  1.40  49.0%  48.3%  2024      1     Winter  0.444444  
3  1.63  49.6%  50.4%  2024      1     Winter  0.222222  
4  1.68  49.9%  48.0%  2024      1     Winter  0.444444  


In [52]:
split_df['split_next'] = split_df['split'] + 1
split_df['year_next'] = split_df['year']

mask = split_df['split'] == 3
split_df.loc[mask, 'split_next'] = 1
split_df.loc[mask, 'year_next'] = split_df.loc[mask, 'year'] + 1

# rename for merging
prior_df = split_df.rename(columns={
    'year_next': 'match_year',
    'split_next': 'match_split',
    'winrate': 'prior_winrate',
    'KD':      'prior_KD',
    'GPR':     'prior_GPR',
    'FBN%':    'prior_FBN',
    'BN%':     'prior_BN',
    'CKPM':    'prior_CKPM',
    'WPM':     'prior_WPM',
    'ELD%':    'prior_ELD',
    'LNE%':    'prior_LNE',
    'JNG%':    'prior_JNG',
})

prior_cols = [
    'teamname', 'match_year', 'match_split',
    'prior_winrate', 'prior_KD', 'prior_GPR',
    'prior_FBN', 'prior_BN', 'prior_CKPM',
    'prior_WPM', 'prior_ELD', 'prior_LNE', 'prior_JNG'
]
prior_df = prior_df[prior_cols]

In [53]:
team_stats = prior_df

In [54]:
team_stats.columns

Index(['teamname', 'match_year', 'match_split', 'prior_winrate', 'prior_KD',
       'prior_GPR', 'prior_FBN', 'prior_BN', 'prior_CKPM', 'prior_WPM',
       'prior_ELD', 'prior_LNE', 'prior_JNG'],
      dtype='object')

In [55]:
team_stats

,teamname,match_year,match_split,prior_winrate,prior_KD,prior_GPR,prior_FBN,prior_BN,prior_CKPM,prior_WPM,prior_ELD,prior_LNE,prior_JNG
0,Fnatic,2024,2,0.666667,1.18,1.04,44%,60%,0.94,3.43,0%,51.4%,54.6%
1,G2 Esports,2024,2,0.666667,1.06,0.21,33%,50%,0.77,3.96,NaN,51.1%,51.4%
2,GiantX,2024,2,0.444444,0.86,-0.72,44%,45%,0.77,3.77,50%,49.0%,48.3%
3,Karmine Corp,2024,2,0.222222,0.74,-0.80,33%,42%,0.83,3.72,100%,49.6%,50.4%
4,MAD Lions KOI,2024,2,0.444444,1.03,0.26,44%,50%,0.84,3.09,NaN,49.9%,48.0%
...,...,...,...,...,...,...,...,...,...,...,...,...,...
77,Natus Vincere,2026,3,0.521739,0.96,0.21,57%,60%,0.81,3.82,50%,49.5%,52.2%
78,Shifters,2026,3,0.272727,0.75,-0.08,41%,43%,0.89,3.29,25%,49.9%,46.7%
79,SK Gaming,2026,3,0.333333,0.67,-0.89,29%,29%,0.89,3.66,0%,50.1%,47.2%
80,Team Heretics,2026,3,0.150000,0.61,-0.94,20%,25%,0.72,3.43,100%,49.2%,46.2%


In [56]:
match_df = pd.concat([
    pd.read_csv('2024_LoL_esports_match_data_from_OraclesElixir.csv'),
    pd.read_csv('2025_LoL_esports_match_data_from_OraclesElixir.csv'),
    pd.read_csv('2026_LoL_esports_match_data_from_OraclesElixir (2).csv'),
], ignore_index=True)



match_df = match_df[
    (match_df['league'] == 'LEC') &
    (match_df['position'] == 'team')
].copy()

match_df['date'] = pd.to_datetime(match_df['date'])
match_df = match_df.sort_values('date').reset_index(drop=True)

print(match_df.shape)
print(match_df.columns.tolist())
print(match_df.head(2))

C:\Users\rares\AppData\Local\Temp\ipykernel_25652\1853790277.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv('2024_LoL_esports_match_data_from_OraclesElixir.csv'),
C:\Users\rares\AppData\Local\Temp\ipykernel_25652\1853790277.py:3: DtypeWarning: Columns (2,5) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv('2025_LoL_esports_match_data_from_OraclesElixir.csv'),
C:\Users\rares\AppData\Local\Temp\ipykernel_25652\1853790277.py:4: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv('2026_LoL_esports_match_data_from_OraclesElixir (2).csv'),


(1692, 165)
['gameid', 'datacompleteness', 'url', 'league', 'year', 'split', 'playoffs', 'date', 'game', 'patch', 'participantid', 'side', 'position', 'playername', 'playerid', 'teamname', 'teamid', 'firstPick', 'champion', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5', 'pick1', 'pick2', 'pick3', 'pick4', 'pick5', 'gamelength', 'result', 'kills', 'deaths', 'assists', 'teamkills', 'teamdeaths', 'doublekills', 'triplekills', 'quadrakills', 'pentakills', 'firstblood', 'firstbloodkill', 'firstbloodassist', 'firstbloodvictim', 'team kpm', 'ckpm', 'firstdragon', 'dragons', 'opp_dragons', 'elementaldrakes', 'opp_elementaldrakes', 'infernals', 'mountains', 'clouds', 'oceans', 'chemtechs', 'hextechs', 'dragons (type unknown)', 'elders', 'opp_elders', 'firstherald', 'heralds', 'opp_heralds', 'void_grubs', 'opp_void_grubs', 'firstbaron', 'barons', 'opp_barons', 'atakhans', 'opp_atakhans', 'firsttower', 'towers', 'opp_towers', 'firstmidtower', 'firsttothreetowers', 'turretplates', 'opp_turretplates', 'in

In [57]:
ROLLING_COLS = [
    'result',
    'golddiffat15', 'xpdiffat15',
    'golddiffat10', 'golddiffat25',
    'firstblood', 'firstdragon', 'firstbaron', 'firsttower', 'firstherald',
    'dragons', 'barons', 'void_grubs',
    'towers', 'inhibitors',
    'kills', 'deaths',
    'gpr', 'vspm', 'wcpm', 'ckpm', 'gamelength',
]

WINDOW = 6

match_df = match_df.sort_values('date').reset_index(drop=True)

rolling_dfs = []
for team, grp in match_df.groupby('teamname'):
    grp = grp.sort_values('date').copy()
    for col in ROLLING_COLS:
        if col in grp.columns:
            grp[f'roll_{col}'] = (
                grp[col].shift(1)
                        .rolling(window=WINDOW, min_periods=2)
                        .mean()
            )
    rolling_dfs.append(grp)

match_df = pd.concat(rolling_dfs).sort_values('date').reset_index(drop=True)
roll_feature_cols = [f'roll_{c}' for c in ROLLING_COLS if c in match_df.columns]

print([c for c in match_df.columns if c.startswith('roll_')])

['roll_result', 'roll_golddiffat15', 'roll_xpdiffat15', 'roll_golddiffat10', 'roll_golddiffat25', 'roll_firstblood', 'roll_firstdragon', 'roll_firstbaron', 'roll_firsttower', 'roll_firstherald', 'roll_dragons', 'roll_barons', 'roll_void_grubs', 'roll_towers', 'roll_inhibitors', 'roll_kills', 'roll_deaths', 'roll_gpr', 'roll_vspm', 'roll_wcpm', 'roll_ckpm', 'roll_gamelength']


In [58]:
prior_stat_cols = ['winrate', 'KD', 'GPR', 'FBN%', 'BN%', 'CKPM', 'WPM', 'ELD%', 'LNE%', 'JNG%']

In [59]:
for col in prior_stat_cols:
    split_df[col] = pd.to_numeric(
        split_df[col].astype(str).str.replace('%', '').str.replace(',', '').str.strip(),
        errors='coerce'
    )


In [60]:
keep_cols = ['gameid', 'date', 'year', 'split', 'playoffs',
             'teamname', 'side', 'result', 'patch'] + roll_feature_cols

blue = (match_df[match_df['side'] == 'Blue'][keep_cols]
        .rename(columns={c: f'blue_{c}' for c in keep_cols if c != 'gameid'})
        .copy())

red  = (match_df[match_df['side'] == 'Red'][keep_cols]
        .rename(columns={c: f'red_{c}' for c in keep_cols if c != 'gameid'})
        .copy())

games = blue.merge(red, on='gameid')

games['blue_win'] = games['blue_result'].astype(int)

games = games.sort_values('blue_date').reset_index(drop=True)

split_name_to_num = {'Winter': 1, 'Versus': 1, 'Spring': 2, 'Summer': 3}
games['split_num'] = games['blue_split'].map(split_name_to_num)

PRIOR_STAT_COLS = ['winrate', 'KD', 'GPR', 'FBN%', 'BN%',
                   'CKPM', 'WPM', 'ELD%', 'LNE%', 'JNG%']

split_df['split_num'] = split_df['split_name'].map(split_name_to_num)

prior_df = split_df[['teamname', 'year', 'split_num'] + PRIOR_STAT_COLS].copy()
prior_df['split_num'] += 1
year_overflow = prior_df['split_num'] > 3
prior_df.loc[year_overflow, 'year']      += 1
prior_df.loc[year_overflow, 'split_num']  = 1

def merge_prior(df, side):
    renamed = prior_df.rename(columns={c: f'{side}_prior_{c}' for c in PRIOR_STAT_COLS})
    return (df.merge(renamed,
                     left_on=[f'{side}_teamname', f'{side}_year', 'split_num'],
                     right_on=['teamname', 'year', 'split_num'],
                     how='left')
              .drop(columns=['teamname', 'year']))

games = merge_prior(games, 'blue')
games = merge_prior(games, 'red')

prior_coverage = games['blue_prior_winrate'].notna().mean()
print(f"Prior stats coverage: {prior_coverage:.1%}")

for col in roll_feature_cols:
    base = col.replace('roll_', '')
    games[f'diff_{base}'] = games[f'blue_{col}'] - games[f'red_{col}']

for col in PRIOR_STAT_COLS:
    games[f'diff_prior_{col}'] = games[f'blue_prior_{col}'] - games[f'red_prior_{col}']

games['blue_side_advantage'] = 1

diff_cols = [c for c in games.columns if c.startswith('diff_')]
feature_cols = diff_cols + ['blue_side_advantage']

X = games[feature_cols].drop(columns=['diff_result'], errors='ignore')
y = games['blue_win']

roll_diff_cols = [c for c in X.columns if 'prior' not in c and c != 'blue_side_advantage']
valid_mask = X[roll_diff_cols].notna().any(axis=1)
X, y = X[valid_mask], y[valid_mask]

X = X.drop(columns=[c for c in X.columns if X[c].isna().mean() > 0.2])
X = X.fillna(0)

print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts()}")
print(f"Feature list:\n{X.columns.tolist()}")

Prior stats coverage: 83.7%
X shape: (827, 31)
y distribution:
blue_win
1    455
0    372
Name: count, dtype: int64
Feature list:
['diff_golddiffat15', 'diff_xpdiffat15', 'diff_golddiffat10', 'diff_golddiffat25', 'diff_firstblood', 'diff_firstdragon', 'diff_firstbaron', 'diff_firsttower', 'diff_firstherald', 'diff_dragons', 'diff_barons', 'diff_void_grubs', 'diff_towers', 'diff_inhibitors', 'diff_kills', 'diff_deaths', 'diff_gpr', 'diff_vspm', 'diff_wcpm', 'diff_ckpm', 'diff_gamelength', 'diff_prior_winrate', 'diff_prior_KD', 'diff_prior_GPR', 'diff_prior_FBN%', 'diff_prior_BN%', 'diff_prior_CKPM', 'diff_prior_WPM', 'diff_prior_LNE%', 'diff_prior_JNG%', 'blue_side_advantage']


In [61]:

K        = 32  
ELO_INIT = 1500  

def expected_score(elo_a, elo_b):
    return 1 / (1 + 10 ** ((elo_b - elo_a) / 400))

def update_elo(elo_a, elo_b, result_a, K=32):
    """result_a: 1 if A won, 0 if A lost"""
    exp_a = expected_score(elo_a, elo_b)
    exp_b = 1 - exp_a
    new_a = elo_a + K * (result_a     - exp_a)
    new_b = elo_b + K * ((1-result_a) - exp_b)
    return new_a, new_b

elo_df = (match_df[match_df['side'] == 'Blue'][['gameid', 'date', 'teamname', 'result']]
          .merge(
              match_df[match_df['side'] == 'Red'][['gameid', 'teamname', 'result']]
              .rename(columns={'teamname': 'red_teamname', 'result': 'red_result'}),
              on='gameid'
          )
          .sort_values('date')
          .reset_index(drop=True))

elo_df = elo_df.rename(columns={'teamname': 'blue_teamname', 'result': 'blue_result'})

elo_ratings = {} 

blue_elos_pre, red_elos_pre = [], []

for _, row in elo_df.iterrows():
    blue_team = row['blue_teamname']
    red_team  = row['red_teamname']

    if blue_team not in elo_ratings:
        elo_ratings[blue_team] = ELO_INIT
    if red_team not in elo_ratings:
        elo_ratings[red_team] = ELO_INIT

    blue_elos_pre.append(elo_ratings[blue_team])
    red_elos_pre.append(elo_ratings[red_team])

    new_blue, new_red = update_elo(
        elo_ratings[blue_team],
        elo_ratings[red_team],
        result_a=row['blue_result'], K=K
    )
    elo_ratings[blue_team] = new_blue
    elo_ratings[red_team]  = new_red

elo_df['blue_elo'] = blue_elos_pre
elo_df['red_elo']  = red_elos_pre
elo_df['diff_elo'] = elo_df['blue_elo'] - elo_df['red_elo']

# ── merge into games ──────────────────────────────────────────────────────────
games = games.merge(
    elo_df[['gameid', 'blue_elo', 'red_elo', 'diff_elo']],
    on='gameid', how='left'
)

# add to feature matrix
X['diff_elo']  = games.loc[X.index, 'diff_elo']
X['blue_elo']  = games.loc[X.index, 'blue_elo']
X['red_elo']   = games.loc[X.index, 'red_elo']

print(games[['blue_teamname', 'red_teamname',
             'blue_elo', 'red_elo', 'diff_elo', 'blue_win']].head(10))
print(f"\nElo range: {elo_df['blue_elo'].min():.0f} – {elo_df['blue_elo'].max():.0f}")
print(f"Final ratings:\n{pd.Series(elo_ratings).sort_values(ascending=False)}")

   blue_teamname   red_teamname  blue_elo  red_elo  diff_elo  blue_win
0       Team BDS     G2 Esports    1500.0   1500.0       0.0         0
1          Rogue      SK Gaming    1500.0   1500.0       0.0         0
2   Karmine Corp         Fnatic    1500.0   1500.0       0.0         0
3  Team Vitality         GiantX    1500.0   1500.0       0.0         0
4  MAD Lions KOI  Team Heretics    1500.0   1500.0       0.0         1
5         GiantX       Team BDS    1516.0   1484.0      32.0         0
6  MAD Lions KOI      SK Gaming    1516.0   1516.0       0.0         0
7     G2 Esports   Karmine Corp    1516.0   1484.0      32.0         1
8         Fnatic  Team Vitality    1516.0   1484.0      32.0         1
9          Rogue  Team Heretics    1484.0   1484.0       0.0         0

Elo range: 1283 – 1795
Final ratings:
G2 Esports           1773.739872
Karmine Corp         1706.819228
Movistar KOI         1642.766489
Team Vitality        1547.810541
GiantX               1539.024524
Fnatic         

In [62]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

In [68]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

def benchmark_models(models, X, y, n_splits=5):
    """
    models = {
        "Logistic Regression": LogisticRegression(),
        "Random Forest": RandomForestClassifier(),
        "XGBoost": XGBClassifier(),
        "CatBoost": CatBoostClassifier(verbose=0)
    }
    """

    tscv = TimeSeriesSplit(n_splits=n_splits)

    results = []

    for name, model in models.items():

        accs = []
        aucs = []
        precisions = []
        recalls = []
        f1s = []

        for train_idx, val_idx in tscv.split(X):

            X_train = X.iloc[train_idx]
            X_val = X.iloc[val_idx]

            y_train = y.iloc[train_idx]
            y_val = y.iloc[val_idx]

            model.fit(X_train, y_train)

            preds = model.predict(X_val)
            probs = model.predict_proba(X_val)[:, 1]

            accs.append(accuracy_score(y_val, preds))
            aucs.append(roc_auc_score(y_val, probs))
            precisions.append(precision_score(y_val, preds))
            recalls.append(recall_score(y_val, preds))
            f1s.append(f1_score(y_val, preds))

        results.append({
            "Model": name,
            "Accuracy": np.mean(accs),
            "ROC_AUC": np.mean(aucs),
            "Precision": np.mean(precisions),
            "Recall": np.mean(recalls),
            "F1": np.mean(f1s)
        })

    leaderboard = (
        pd.DataFrame(results)
        .sort_values("ROC_AUC", ascending=False)
        .reset_index(drop=True)
    )

    print("\n=== LEADERBOARD ===\n")
    print(leaderboard.round(4))

    print("\n=== README TABLE ===\n")
    print("| Rank | Model | ROC-AUC | Accuracy | Precision | Recall | F1 |")
    print("|------|--------|---------|----------|-----------|--------|----|")

    medals = ["🥇", "🥈", "🥉"]

    for i, row in leaderboard.iterrows():

        rank = medals[i] if i < 3 else str(i + 1)

        print(
            f"| {rank} "
            f"| {row['Model']} "
            f"| {row['ROC_AUC']:.4f} "
            f"| {row['Accuracy']:.4f} "
            f"| {row['Precision']:.4f} "
            f"| {row['Recall']:.4f} "
            f"| {row['F1']:.4f} |"
        )

    return leaderboard

In [69]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(
        eval_metric="logloss",
        random_state=42
    ),
    "CatBoost": CatBoostClassifier(
        verbose=0,
        random_seed=42
    ),
    "CatBoost + Optuna": CatBoostClassifier(
        **study.best_params,
        verbose=0,
        random_seed=42
    )
}

leaderboard = benchmark_models(models, X, y)

C:\Program Files\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Program Files\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_i


=== LEADERBOARD ===

                 Model  Accuracy  ROC_AUC  Precision  Recall      F1
0    CatBoost + Optuna    0.6453   0.7031     0.6546  0.7780  0.7059
1        Random Forest    0.6394   0.6782     0.6644  0.7110  0.6843
2             CatBoost    0.6394   0.6759     0.6577  0.7316  0.6904
3  Logistic Regression    0.5898   0.6323     0.6186  0.6713  0.6414
4              XGBoost    0.5898   0.6308     0.6216  0.6660  0.6411

=== README TABLE ===

| Rank | Model | ROC-AUC | Accuracy | Precision | Recall | F1 |
|------|--------|---------|----------|-----------|--------|----|
| 🥇 | CatBoost + Optuna | 0.7031 | 0.6453 | 0.6546 | 0.7780 | 0.7059 |
| 🥈 | Random Forest | 0.6782 | 0.6394 | 0.6644 | 0.7110 | 0.6843 |
| 🥉 | CatBoost | 0.6759 | 0.6394 | 0.6577 | 0.7316 | 0.6904 |
| 4 | Logistic Regression | 0.6323 | 0.5898 | 0.6186 | 0.6713 | 0.6414 |
| 5 | XGBoost | 0.6308 | 0.5898 | 0.6216 | 0.6660 | 0.6411 |


In [40]:
import optuna
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score

tscv = TimeSeriesSplit(n_splits=5)

def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 800),
        "depth": trial.suggest_int("depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "loss_function": "Logloss",
        "verbose": 0,
        "random_seed": 42,
    }

    auc_scores = []

    for train_idx, val_idx in tscv.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = CatBoostClassifier(**params)
        model.fit(X_train, y_train)

        preds = model.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, preds))

    return np.mean(auc_scores)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(study.best_params)

[I 2026-06-13 21:18:07,641] A new study created in memory with name: no-name-6fa61281-d6ce-4b12-90b2-df943443ed1e
[I 2026-06-13 21:18:10,871] Trial 0 finished with value: 0.6543615102672862 and parameters: {'iterations': 307, 'depth': 5, 'learning_rate': 0.04185923284646321, 'l2_leaf_reg': 1.043169947830218, 'random_strength': 1.484162927335416, 'bagging_temperature': 0.11342470524013937, 'border_count': 35}. Best is trial 0 with value: 0.6543615102672862.
[I 2026-06-13 21:18:15,542] Trial 1 finished with value: 0.672213377572989 and parameters: {'iterations': 739, 'depth': 2, 'learning_rate': 0.010849690685736216, 'l2_leaf_reg': 3.2157038767405823, 'random_strength': 0.21955047133801409, 'bagging_temperature': 0.25749519440223, 'border_count': 173}. Best is trial 1 with value: 0.672213377572989.
[I 2026-06-13 21:18:27,381] Trial 2 finished with value: 0.6707100136793407 and parameters: {'iterations': 411, 'depth': 7, 'learning_rate': 0.09395738159271808, 'l2_leaf_reg': 0.2609128774862

{'iterations': 101, 'depth': 2, 'learning_rate': 0.03430066384839853, 'l2_leaf_reg': 0.17059548582638917, 'random_strength': 1.3444351565329766, 'bagging_temperature': 0.9035943582958805, 'border_count': 252}
